# dataset.ipynb — Pipeline de données
Transformations, augmentation, DataLoaders pour Chest X-Ray.

In [ ]:
import os
from pathlib import Path
from typing import Dict, Tuple
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

IMAGE_SIZE   = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
CLASS_NAMES   = ['NORMAL', 'PNEUMONIA']

print('Librairies importees.')

## Transformations (train vs val/test)

In [ ]:
def get_transforms(mode: str = 'train') -> transforms.Compose:
    if mode == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

print('Train transforms :', get_transforms('train'))
print('\nVal transforms   :', get_transforms('val'))

## Datasets et DataLoaders

In [ ]:
def get_datasets(data_dir: str = '../data/chest_xray') -> Dict:
    base = Path(data_dir)
    splits = {'train': base/'train', 'val': base/'val', 'test': base/'test'}
    datasets_dict = {}
    for split, path in splits.items():
        if not path.exists():
            raise FileNotFoundError(f'Repertoire introuvable : {path}')
        dataset = datasets.ImageFolder(root=str(path), transform=get_transforms(mode=split))
        datasets_dict[split] = dataset
        print(f'  [{split:5s}] {len(dataset):5d} images  |  classes : {dataset.class_to_idx}')
    return datasets_dict

def get_dataloaders(data_dir='../data/chest_xray', batch_size=32, num_workers=0):
    dsets = get_datasets(data_dir)
    train_loader = DataLoader(dsets['train'], batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(dsets['val'],   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=False)
    test_loader  = DataLoader(dsets['test'],  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=False)
    return train_loader, val_loader, test_loader

## Test du pipeline

In [ ]:
print('=== Verification du dataset ===')
train_dl, val_dl, test_dl = get_dataloaders('../data/chest_xray')
batch_imgs, batch_labels = next(iter(train_dl))
print(f'\nForme batch : {batch_imgs.shape}   labels : {batch_labels.shape}')
print(f'  min={batch_imgs.min().item():.3f}  max={batch_imgs.max().item():.3f}')
print('Dataset pret - OK')